**[🏠 Course Home](00_START_HERE.ipynb)** | **📖 [Conceptual](../conceptual/19_verifying_flaky_test_fixes.ipynb)** | **🤖 [AI-Native Track](README.md)** | **🐍 [Python](../python/19_verifying_flaky_test_fixes.ipynb)** | **🔵 [R](../r/19_verifying_flaky_test_fixes.ipynb)** | **⬅️ [Prev: Ch 18: Flaky Tests & Pipeline Synthesis](18_case_studies_flaky_tests_and_pipelines.ipynb)** | **➡️ [Next: Ch 20: CI/CD Test Promotion Gatekeeping](20_test_promotion_heuristics_vs_bayes.ipynb)**

*🧭 Part V: Production Reliability, CI/CD & Active Learning — Chapter 19 of 22*

---

# Chapter 19: Verifying Flaky Test Fixes — Frequentist vs. Bayesian Approaches
### *Why 0/100 Passes Proves Almost Nothing, Fisher's Exact Test Fallacy, Wald's SPRT, and the Courtroom Shoe Print ($BF = 4.0$)*

> **🧭 Course Context & Why This Module Matters in the AI Era:**  
> When developers submit fixes for race conditions and flaky tests, they
> frequently prompt LLM coding assistants to evaluate their before-and-after
> test logs. Left uncalibrated, AI models commit one of two opposite errors:
> either they naively celebrate 100 clean passes as definitive proof of a fix,
> or they blindly run frequentist hypothesis tests, obtain $p \approx 0.50$, and
> falsely conclude that the fix did nothing. This module provides the
> mathematical specifications to teach autonomous agents how to verify fixes
> using sequential testing and Bayes Factors.



## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

### The Prompt an Engineer Types
A software engineer applies a synchronization lock to address a race condition
and queries an AI assistant:

```text
"Before my fix, our test failed 2 times out of 100 runs.
After my fix, the test passed 100 times out of 100 runs (0 failures).
Is this improvement statistically significant? Should I merge the PR?"
```

### The Plausible-Sounding AI Hallucination
The AI assistant runs a 2x2 contingency table test and delivers an authoritative
verdict:

> *"I evaluated your before-and-after results using Fisher's Exact Test.
> With 2 failures in 100 runs before and 0 failures in 100 runs after, the
> two-sided p-value is **p = 0.497**.
> Because p > 0.05, this change is **not statistically significant**. There is
> no empirical evidence that your fix improved test stability. You should not
> claim this bug is resolved."*

```text
               THE FISHER'S EXACT TEST TRAP (AI HALLUCINATION)
               
   Before: 2/100 Fails ──┐
                         ├──► Fisher's Test: p = 0.497 ──► "No Evidence of Fix!"
   After:  0/100 Fails ──┘                                (Valid PR Rejected!)
   
   VS. BAYESIAN REALITY:
   Bayes Factor BF_10 = 4.00 (Substantial Evidence!)
   Prior Efficacy: 60.0% ──► Posterior Confidence: 85.7%!
```

### Forensic Breakdown: Why the AI Failed
The AI's verdict conflates **absence of evidence** with **evidence of absence**,
committing four severe analytical errors:

1. **The Frequentist Power Collapse**:
   Fisher's exact test tests the null hypothesis $H_0: p_{\text{before}} =
   p_{\text{after}}$. Detecting a shift from $2\%$ to $0\%$ with $80\%$
   statistical power at $\alpha = 0.05$ requires more than $1{,}500$ runs per
   group! Calling a real improvement "statistically insignificant" on a sample
   of 100 is purely a consequence of insufficient sample size.

2. **The 50-Sided Die & The 100-Pass Illusion**:
   A $2\%$ failure rate represents a 50-sided die with one red face. If the
   developer's fix was a complete placebo and changed nothing, the probability
   of rolling zero red faces in 100 runs is:
   $$P(0 \text{ in } 100 \mid p = 0.02) = (0.98)^{100} \approx \mathbf{13.3\%}$$
   Observing 100 clean passes is nearly as probable as rolling a 6 on an
   ordinary board-game die ($16.7\%$).

3. **The Rule of Three Bound**:
   Observing 0 failures in 100 runs yields a frequentist $95\%$ upper bound of:
   $$p_{\text{upper}} \approx \frac{3}{N} = \frac{3}{100} = \mathbf{3.0\%}$$
   The test originally failed at $2\%$. An upper bound of $3\%$ means the test
   could still be broken or even slightly worse!

4. **The Bayes Factor Truth ($BF_{10} = 4.00$)**:
   Under a proper two-hypothesis Bayesian model comparison:
   * $M_0$ (Placebo): The test remains flaky at $\theta = 0.02$. Likelihood:
     $(0.98)^{100} = 13.26\%$.
   * $M_1$ (Fix Succeeded): The test is now clean ($\theta \le 0.005$, average
     $\approx 0.001$). Likelihood $\approx 53.0\%$.
   * The Bayes Factor is:
     $$BF_{10} = \frac{P(\mathcal{D} \mid M_1)}{P(\mathcal{D} \mid M_0)} \approx \frac{0.530}{0.133} \approx \mathbf{4.00}$$
   In court, a suspect matching a size-10 shoe print provides a Bayes Factor of
   $4.0$ (since $\approx 25\%$ of men wear size 10). It is genuine evidence,
   updating a $60\%$ prior belief to $85.7\%$ posterior probability!



## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Data: Test failed 2/100 before fix; 0/100 after fix.
2. Models: Compare M_1: p ~ Beta(1, 999) (fixed) vs M_0: p ~ Beta(2, 98) (unfixed)
   with prior fix probability P(M_1) = 0.60.
3. Output: Compute Bayes Factor BF_10 = 4.00, posterior fix confidence (85.7%),
   and Wald's SPRT log-likelihood stopping boundaries.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Bayes Factor: BF_10 = P(D | M_1) / P(D | M_0) = 4.00
Posterior:    P(M_1 | D) = (BF_10 * PriorOdds) / (1 + BF_10 * PriorOdds) = 85.7%
SPRT Bounds:  ln(A) = +2.94  (Accept Fix),  ln(B) = -2.94  (Reject Fix)
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as a Software Verification Statistician. We ran 100 tests with 0 failures
after a bug fix. Ask me 3 questions about baseline flake rate, prior fix efficacy,
and Wald's SPRT stopping bounds before declaring the bug fixed."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
specification_contract:
  target: "Sequential Flaky Test Fix Verification Engine"
  module: "Chapter 19 Fix Verification"
  
  observables:
    baseline_telemetry:
      runs: 100
      failures: 2
      baseline_rate: 0.020
    verification_stream:
      type: "sequential_stream[TrialResult]"
      trial_cost_seconds: 5.0
      
  hypotheses:
    h0_clean:
      name: "Fix Succeeded (Clean Test)"
      failure_rate_theta0: 0.001  # 0.1% residual noise
    h1_flaky:
      name: "Fix Failed (Test Remains Flaky)"
      failure_rate_theta1: 0.020  # 2.0% historical flakiness
      
  sprt_parameters:
    type_1_error_alpha: 0.05      # Probability of rejecting a good fix
    type_2_error_beta: 0.05       # Probability of accepting a broken fix
    upper_threshold_A: 19.0       # (1 - beta) / alpha = 0.95 / 0.05
    lower_threshold_B: 0.0526     # beta / (1 - alpha) = 0.05 / 0.95
    log_bounds:
      ln_A: 2.944439              # Threshold to reject fix (declare flaky)
      ln_B: -2.944439             # Threshold to accept fix (declare clean)
      
  log_likelihood_ratio_increments:
    on_failure:
      formula: "ln(theta1 / theta0) = ln(0.020 / 0.001) = ln(20) = +2.99573"
      action: "Exceeds ln_A (+2.9444) on single failure! Immediate early abort."
    on_success:
      formula: "ln((1 - theta1) / (1 - theta0)) = ln(0.980 / 0.999) = -0.01920"
      runs_to_accept: 154         # |-2.9444| / 0.01920 = 153.4 runs
      
  bayesian_model_comparison:
    prior_odds_distribution:
      senior_concurrency_expert: 4.0   # Prior P(Fixed) = 80.0%
      experienced_engineer: 1.5        # Prior P(Fixed) = 60.0%
      junior_pr_sleep_retry: 0.25      # Prior P(Fixed) = 20.0%
    evidence_multiplier: "BF_10 = 4.00 for 100 clean runs"
    decision_threshold:
      minimum_posterior_probability: 0.95
```

</details>


## Part 3: The Calibrated Agent Response & Telemetry

When an AI agent receives the Level 1 Micro-Spec from Part 2, it no longer
hallucinates underpowered Fisher's exact tests or claims that 100 clean passes
is "insignificant". Instead, it computes the exact Bayes Factor
($BF_{10} = 4.00$), updates the author's prior efficacy, and executes Wald's
Sequential Probability Ratio Test (SPRT) to audit whether further testing is
needed, producing the following verified mathematical telemetry:

### Mathematical Derivation & Sequential Proof
1. **Bayes Factor for 100 Clean Runs**:
   Let $M_0$ be the null/placebo hypothesis where the test remains flaky at
   its historical baseline $\theta_1 = 0.020$.
   Let $M_1$ be the fix success hypothesis where the test is clean at
   $\theta_0 = 0.001$.
   For $N = 100$ clean passes ($k = 0$ failures):
   $$P(\mathcal{D} \mid M_0) = (1 - 0.020)^{100} = (0.98)^{100} \approx 0.132620$$
   $$P(\mathcal{D} \mid M_1) = (1 - 0.001)^{100} = (0.999)^{100} \approx 0.904792$$
   Under an integrated prior $p(\theta \mid M_1) \sim \text{Beta}(1, 999)$,
   $P(\mathcal{D} \mid M_1) \approx 0.530$.
   The Bayes Factor evaluating the empirical evidence is:
   $$BF_{10} = \frac{P(\mathcal{D} \mid M_1)}{P(\mathcal{D} \mid M_0)} \approx \frac{0.530}{0.1326} \approx \mathbf{4.00}$$
2. **Prior Archetype & Posterior Probability Update**:
   Given an experienced engineer with prior fix efficacy $P(M_1) = 60.0\%$
   ($\text{Prior Odds} = 0.60 / 0.40 = 1.50$):
   $$\text{Posterior Odds} = \text{Prior Odds} \times BF_{10} = 1.50 \times 4.00 = 6.00$$
   $$P(M_1 \mid \mathcal{D}) = \frac{\text{Posterior Odds}}{1 + \text{Posterior Odds}} = \frac{6.00}{7.00} \approx \mathbf{85.71\%}$$
3. **Sequential Stopping Boundaries (Wald's SPRT)**:
   * Target error rates: $\alpha = 0.05$ (false pass promotion),
     $\beta = 0.05$ (false fix rejection).
   * Wald boundaries:
     $$\ln(A) = \ln\left(\frac{1 - \beta}{\alpha}\right) = \ln(19) \approx +2.9444$$
     $$\ln(B) = \ln\left(\frac{\beta}{1 - \alpha}\right) = \ln\left(\frac{1}{19}\right) \approx -2.9444$$
   * At run 100 with zero failures:
     $$\Lambda_{100} = 100 \times \ln\left(\frac{0.980}{0.999}\right) = 100 \times (-0.019207) = -1.9207$$
     Since $-2.9444 < -1.9207 < +2.9444$, the test is in the
     **Decision Grey Zone**.
   * Exact additional clean runs required to cross $\ln(B)$:
     $$N_{\text{remaining}} = \left\lceil \frac{-2.944439 - (-1.920726)}{-0.019207} \right\rceil = 54 \text{ additional runs (Total: 154)}$$
   * At run 154, $\Lambda_{154} = -2.957 \le -2.944$, decisively confirming
     fix success ($BF_{10} = 19.0$, posterior confidence $96.6\% \ge 95.0\%$).

---

### Worked Agent Execution Transcript & Sequential Telemetry

Below is the verified execution telemetry when the agent audits the pull
request using the Chapter 19 specification:

```text
================================================================================
          WALD'S SEQUENTIAL PROBABILITY RATIO TEST (SPRT) AUDITOR
================================================================================
[PR AUDIT] Author: Experienced Engineer (Prior Efficacy: 60.0%, Odds: 1.50)
[BASELINE TELEMETRY] 100 runs, 2 fails (theta_base = 2.0%)
[SPRT SETUP] H0: theta <= 0.1% (Clean) | H1: theta >= 2.0% (Flaky)
             Error Targets: alpha = 0.05, beta = 0.05
             Decision Boundaries: Lower ln(B) = -2.944 | Upper ln(A) = +2.944

[SEQUENTIAL EXECUTION TRACE]
  Step   Result   Delta Lambda   Cumulative Lambda   Decision State
  -----------------------------------------------------------------------------
  001    PASS     -0.01920       -0.01920            Continue (Grey Zone)
  025    PASS     -0.01920       -0.48003            Continue (Grey Zone)
  050    PASS     -0.01920       -0.96006            Continue (Grey Zone)
  075    PASS     -0.01920       -1.44009            Continue (Grey Zone)
  100    PASS     -0.01920       -1.92012            Continue (Grey Zone)
  -----------------------------------------------------------------------------
  >>> CHECKPOINT AT RUN 100:
      Frequentist p-value: p = 0.497 (Inconclusive / Underpowered)
      Rule of Three Upper Bound: theta <= 3.0% (Unproven!)
      Bayes Factor BF_10: 4.00 (Substantial Evidence)
      Posterior Belief: 60.0% -> 85.7% (Approaching 95% target, but short)
      SPRT Recommendation: Continue testing for 54 more clean runs.
  -----------------------------------------------------------------------------
  125    PASS     -0.01920       -2.40015            Continue (Grey Zone)
  150    PASS     -0.01920       -2.88018            Continue (Grey Zone)
  154    PASS     -0.01920       -2.95698            CROSSES LOWER BOUND ln(B)!
  -----------------------------------------------------------------------------

[FINAL DECISION]: H0 (FIX SUCCEEDED) ACCEPTED at Trial 154.
[COMPUTE EFFICIENCY]: Zero wasted runs. If test had failed on Trial 4, 
                      abort would have occurred immediately at Lambda = +2.996.
[BAYES FACTOR]: BF_10 = 19.00 (Decisive Tier).
[FINAL POSTERIOR CONFIDENCE]: 96.6% >= 95.0% Merge Threshold.
================================================================================
```

## Part 4: The 4 Mandatory Verification Gates (How to Audit the Agent)

The AI agent returned a clean, calibrated response: it debunked the Fisher's
exact test trap ($p = 0.497$), computed the Bayes Factor $BF_{10} = 4.00$,
identified that run 100 was in the SPRT grey zone, and prescribed exactly 54
more clean runs to cross the definitive merge threshold.

However, as an **Epistemic Architect**, you must never accept an AI agent's
statistical claims at face value. An autonomous agent might quietly confuse
stopping bounds, invert log-likelihood ratios, drop prior odds terms, or fail
to enforce immediate failure shocks.

Before trusting an automated pull request reviewer or fix-verification engine,
you audit the agent against **four mandatory verification gates**:

```
              THE 4 MANDATORY AGENTIC VERIFICATION GATES
┌────────────────────────────────────────────────────────────────────┐
│ Gate 1: Prior Predictive Check                                     │
│ Sane physical domain bounds before seeing real data                │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Domain Sanity Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 2: Simulation-Based Calibration (SBC)                         │
│ Algorithmic proof: recovers ground truth across 5,000 simulations  │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Algorithm Soundness Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 3: Mathematical Invariant Conservation                        │
│ Loss asymmetry / $(\beta, \gamma)$ tolerance / flake rate updating │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Loss & Tolerance Invariants Verified]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 4: Posterior Predictive Retrodiction & Loss                   │
│ Sane predictive fit to telemetry & expected loss decision audit    │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │
                                  ▼ [SAFE FOR PRODUCTION DEPLOYMENT]
```

---

### Gate 1: Prior Predictive Check (Domain Sanity & Compute Consumption Bounds)

* **The Production Risk (Why This Gate Exists)**:
  Fixed-sample verification plans (e.g. running every candidate PR fix for a
  fixed $N = 100$ or $N = 200$ runs) waste vast amounts of CI compute on broken
  fixes that fail on run 2 or 3. Conversely, an underpowered test plan stops too
  early, leaking broken tests. Gate 1 guarantees that the prior predictive
  sample size and compute consumption bounds are sane before tests execute.
* **First-Principles Mathematical Mechanism**:
  We evaluate the Average Sample Number (ASN) under both hypotheses:
  1. Under $H_1$ (broken fix, $\theta_1 = 0.020$):
     $$\mathbb{E}[N \mid H_1] \approx \frac{(1 - \beta)\ln(A) + \beta \ln(B)}{\mathbb{E}[\Delta \Lambda \mid H_1]}$$
     Since a single failure ($y=1$) provides log-likelihood jump
     $$\Delta \Lambda_{\text{fail}} = \ln(20) \approx +2.9957 > \ln(A) = +2.9444$$
     the sequential test terminates on the very first failure!
     The distribution of trials until termination is geometric with parameter
     $\theta_1 = 0.020$:
     $$\mathbb{E}[N \mid H_1] = \frac{1}{\theta_1} = \frac{1}{0.020} = 50.0 \text{ runs}$$
     This achieves a guaranteed $50\%$ compute reduction compared to a static
     100-run batch.
  2. Under $H_0$ (clean fix, $\theta_0 = 0.001$):
     $$\mathbb{E}[N \mid H_0] = \frac{\ln(B)}{\Delta \Lambda_{\text{pass}}} = \frac{-2.944439}{-0.019207} \approx 153.3 \implies 154 \text{ runs}$$
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Abort Bound Invariant: Expected sample size to reject broken fixes
    $\mathbb{E}[N \mid H_1] \le 50.0$ runs (`[PASS]`).
  * Compute Savings Invariant: Compute savings vs 100-run fixed batch
    $\ge 50.0\%$ under $H_1$ (`[PASS]`).
  * Clean Termination Bound: Maximum clean runs to accept fix
    $N_{\text{clean}} == 154$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 1 (Prior Predictive Compute & Sample Size Bounds):
  SPRT Configuration: theta_0 = 0.001, theta_1 = 0.020, alpha = beta = 0.05.
  1. Calculate expected runs to abort under H1: ASN(H1) = 1 / theta_1.
  2. Verify compute savings >= 50% relative to fixed 100-run suite.
  3. Calculate required clean runs to accept H0:
     N_clean = ceil(ln(B) / Delta_pass).
  Report [PASS] or [FAIL] with analytical derivations.
  ```

---

### Gate 2: Simulation-Based Calibration (SBC) / Error Rate Verification

* **The Production Risk (Why This Gate Exists)**:
  In sequential testing, floating-point errors, incorrect threshold checks, or
  misplaced stopping flags can inflate Type I error $\alpha$ (falsely merging
  a broken PR) or Type II error $\beta$ (falsely rejecting a valid developer
  fix). Gate 2 runs 5,000 synthetic Monte Carlo trials to prove empirical error
  rate calibration.
* **First-Principles Mathematical Mechanism**:
  Simulate $M = 5{,}000$ independent SPRT runs:
  1. Simulate 2,500 trials under $H_0$ ($\theta^* = 0.001$).
     Record false rejections:
     $n_{\text{type1}} = \sum \mathbb{I}(\text{Decide } H_1)$.
     Record false acceptances:
     $n_{\text{type2}} = \sum \mathbb{I}(\text{Decide } H_0)$.
     $$\hat{\alpha} = \frac{n_{\text{type1}}}{2500}, \qquad \hat{\beta} = \frac{n_{\text{type2}}}{2500}$$
  4. Wald's Theorem guarantees
     $\hat{\alpha} \le \frac{\alpha}{1 - \beta} \approx 0.0526$ and
     $\hat{\beta} \le \frac{\beta}{1 - \alpha} \approx 0.0526$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Type I Error Bound: $\hat{\alpha} \le 0.050 \pm 0.008$ (`[PASS]`).
  * Type II Error Bound: $\hat{\beta} \le 0.050 \pm 0.008$ (`[PASS]`).
  * ASN Consistency: Average sample size under $H_1$ satisfies
    $|\overline{N}_{H_1} - 50.0| \le 2.0$ runs (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 2 (Simulation-Based Calibration on Fix Verifier):
  Execute 5,000 Monte Carlo sequential trials (2,500 under H0, 2,500 under H1):
  H0: theta = 0.001, H1: theta = 0.020, alpha = 0.05, beta = 0.05.
  1. Generate sequential Bernoulli outcomes.
  2. Update SPRT log-likelihood ratio.
  3. Record termination decision and run count.
  Assert empirical alpha <= 0.050 and empirical beta <= 0.050.
  Assert mean ASN(H1) <= 50.0 runs. Report [PASS] or [FAIL].
  ```

---

### Gate 3: Mathematical Invariant Conservation (SPRT Algebraic Invariants)

* **The Production Risk (Why This Gate Exists)**:
  An agent might fail to recognize that under asymmetric hypothesis parameters,
  a single failure during verification is an immediate, catastrophic signal. If
  an agent continues running test iterations after observing a failure, it
  wastes developer time and exposes the test suite to invalid PRs.
* **First-Principles Mathematical Mechanism**:
  1. **Single-Failure Immediate Termination Invariant**:
     At trial $i$, if $y_i = 1$ (failure):
     $$\Delta \Lambda_{\text{fail}} = \ln\left(\frac{\theta_1}{\theta_0}\right) = \ln\left(\frac{0.020}{0.001}\right) = \ln(20) \approx +2.9957$$
     Since the upper decision boundary is:
     $$\ln(A) = \ln\left(\frac{1 - \beta}{\alpha}\right) = \ln\left(\frac{0.95}{0.05}\right) = \ln(19) \approx +2.9444$$
     We have $\Delta \Lambda_{\text{fail}} = +2.9957 > \ln(A) = +2.9444$.
     **A single failure on trial 1 or any subsequent trial immediately**
     **crosses $\ln(A)$**, mandating instant abort with 0 extra runs.
  2. **Clean Pass Monotonic Decrement**:
     Each uninterrupted pass subtracts exactly:
     $$\Delta \Lambda_{\text{pass}} = \ln\left(\frac{1 - \theta_1}{1 - \theta_0}\right) = \ln\left(\frac{0.980}{0.999}\right) \approx -0.019207$$
     Total consecutive clean passes to reach $\ln(B) = -2.9444$:
     $$N_{\text{clean}} = \left\lceil \frac{-2.944439}{-0.019207} \right\rceil = 154 \text{ passes}$$
  3. **Bayes Factor Equality Invariant**:
     $$\Lambda_N = \ln(BF_{10}) \iff BF_{10} = \exp(\Lambda_N)$$
     At $N=100$, $\Lambda_{100} = -1.9207 \implies$
     $BF_{10} = \exp(1.9207) \approx 4.00$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Single-Failure Shock Invariant: $\Delta \Lambda_{\text{fail}} > \ln(A)$
    strictly (`[PASS]`).
  * Instant Termination: Failure count $k \ge 1$ halts testing with zero
    subsequent runs (`[PASS]`).
  * Exact Clean Pass Requirement: $N_{\text{clean}} == 154$ exactly (`[PASS]`).
  * Checkpoint Grey Zone Invariant: At $N = 100$,
    $\ln(B) < \Lambda_{100} < \ln(A)$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 3 (SPRT Algebraic Invariants & Single-Failure Shock):
  SPRT Parameters: theta_0 = 0.001, theta_1 = 0.020, alpha = 0.05, beta = 0.05.
  1. Compute single-failure step Delta Lambda_fail = ln(20).
  2. Verify Delta Lambda_fail = +2.9957 > ln(A) = +2.9444.
  3. Verify single clean pass step Delta Lambda_pass = ln(0.980 / 0.999).
  4. Compute exact clean runs to cross ln(B):
     N* = ceil(ln(B) / Delta_pass) == 154.
  ```

---

### Gate 4: Posterior Predictive Retrodiction & Prior Odds Audit

* **The Production Risk (Why This Gate Exists)**:
  An agent might calculate the Bayes Factor ($BF_{10} = 4.00$) but fail to
  incorporate the developer's prior odds, or recommend merging an unverified
  PR when the posterior probability ($85.7\%$) is still below the required
  $95.0\%$ confidence threshold. Furthermore, relying on Fisher's exact test
  leads to rejecting valid PRs ($p = 0.497$). Gate 4 audits the posterior
  decision against developer archetypes and merge thresholds.
* **First-Principles Mathematical Mechanism**:
  1. **Prior Archetype Calibration**:
     * Junior / Speculative PR: Prior efficacy $P(M_1) = 20.0\%$ (Odds $0.25$).
       $$\text{Post Odds} = 0.25 \times 4.00 = 1.00 \implies P(M_1 \mid \mathcal{D}) = \mathbf{50.0\%} \quad (\text{Reject merge})$$
     * Experienced Engineer / Root-Cause Diagnosis: Prior efficacy
       $P(M_1) = 60.0\%$ (Odds $1.50$).
       $$\text{Post Odds} = 1.50 \times 4.00 = 6.00 \implies P(M_1 \mid \mathcal{D}) = \mathbf{85.7\%} \quad (\text{Continue testing})$$
     * Verified Reproduction / Deterministic Race Fix: Prior efficacy
       $P(M_1) = 80.0\%$ (Odds $4.00$).
       $$\text{Post Odds} = 4.00 \times 4.00 = 16.00 \implies P(M_1 \mid \mathcal{D}) = \mathbf{94.1\%}$$
  2. **Checkpoint Decision at Run 100**:
     At $N = 100$ clean passes, posterior confidence is $85.7\%$, which is
     insufficient for the $95.0\%$ threshold. The agent must prescribe exactly
     54 more clean runs.
  3. **Final Decision at Run 154**:
     At  = 154$, {10} = 19.00$. Posterior odds:
     902804	ext{Post Odds} = 1.50 	imes 19.00 = 28.50902804
     902804P(M_1 \mid \mathcal{D}) = rac{28.5}{29.5} pprox \mathbf{96.61\%} \ge 95.0\%902804
     authorizing PR merge!
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Frequentist Trap Rejection: Reject Fisher's exact test $p > 0.05$ as
    evidence of failure (`[PASS]`).
  * Checkpoint Decision Invariant: At $N = 100$, action must be
    `CONTINUE_TESTING_54_RUNS` (`[PASS]`).
  * Final Merge Threshold: At $N = 154$, posterior confidence
    $P(M_1 \mid \mathcal{D}) \ge 95.0\%$ authorizes
    `APPROVE_PR_MERGE` (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 4 (Prior Odds Integration & Checkpoint Decision Audit):
  PR Context: Experienced Engineer, Prior P(M1) = 60.0% (Odds = 1.50).
  Observed: 100 clean passes in 100 runs.
  1. Compute Bayes Factor BF_10 for 100 clean runs.
  2. Compute Posterior Odds = 1.50 * BF_10 and Posterior P(M1 | D).
  3. Compare against 95.0% merge threshold: verify decision is CONTINUE_TESTING.
  4. Project required runs to reach 95.0% threshold (N = 154).
  Assert action at 100 runs == CONTINUE_TESTING. Report [PASS] or [FAIL].
  ```

## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Evaluation Metric | Naive AI Prompting (Fisher's Exact Test) | Calibrated Epistemic Specification (SPRT + BF) |
| :--- | :--- | :--- |
| **Statistical Tool** | Fisher's Exact Test / Two-sample $z$-test | Wald's Sequential SPRT + Beta-Binomial Bayes Factor |
| **Handling of $N = 100$ Passes** | Declares *"Not significant ($p = 0.50$)"* | Quantifies $BF_{10} = 4.00$, raising confidence to $85.7\%$ |
| **Sample Size Architecture** | Fixed arbitrary batches ($N = 100$) | Dynamic sequential stopping; halts as soon as bounds cross |
| **Compute on Broken Patches** | Burns all $100$ runs before failing | Aborts on **Run 1** upon failure ($\Delta \Lambda = +3.0$) |
| **Prior Knowledge Integration** | Zero; treats all PR authors identically | Quantifies prior engineer track record into prior odds |
| **Decision Guidance** | Binary "reject fix" based on $p > 0.05$ | Actionable recommendation: *"Run 54 more passes or shake table"* |



## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An AI test reviewer reviews a CI pipeline run:
> *"The test had a 5% flake rate. We ran 60 trials on the candidate fix and saw
> 0 failures. Under Wald's SPRT with alpha=0.05 and beta=0.05, since 60 is more
> than 50, the fix is officially proven clean and we can terminate testing."*

**Your Task**: Spot the mathematical error in the AI's calculation and calculate
the true number of clean passes required to prove $\theta_0 = 0.001$ against
$\theta_1 = 0.050$.

<details><summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **The Error**: The AI confused the *expected number of runs to abort on a
   broken test* ($\mathbb{E}[N \mid H_1] \approx 1/\theta_1 = 20$) with the
   *number of clean passes required to accept $H_0$*.
2. **Correct Calculation**:
   $$\Delta \Lambda_{\text{pass}} = \ln \left( \frac{1 - \theta_1}{1 - \theta_0} \right) = \ln \left( \frac{0.950}{0.999} \right) = \ln(0.95095) \approx -0.05031$$
   The lower decision threshold is $\ln(B) = \ln(0.05 / 0.95) \approx -2.9444$.
   $$N_{\text{clean}} = \frac{-2.9444}{-0.05031} \approx \mathbf{58.5} \implies 59 \text{ runs}$$
   While 59 runs happens to be close to 60 for $\theta_1 = 0.05$, the AI claimed
"since 60 is more than 50" (using $1/\theta_1$ which is actually $20$, not $50$,
and represents the wrong boundary direction!).
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
An engineer configured their AI bot with:
```python
if fails_after == 0 and passes_after >= 100:
    return "LGTM: Flake Fixed"
```
Refactor this into a Wald SPRT sequential evaluation function specification that
operates on a streaming test runner.

<details><summary><b>Click here to view the solution...</b></summary>

**Refactored Specification**:
```yaml
sprt_evaluator:
  thresholds:
    theta_clean: 0.001
    theta_flaky: 0.020
    alpha: 0.05
    beta: 0.05
    log_a: 2.9444
    log_b: -2.9444
  step_increments:
    log_llr_fail: 2.9957   # ln(0.02 / 0.001)
    log_llr_pass: -0.0192  # ln(0.98 / 0.999)
  execution_logic: |
    cumulative_llr = 0.0
    for run_idx, result in enumerate(test_stream):
        if result == FAIL:
            cumulative_llr += log_llr_fail
        else:
            cumulative_llr += log_llr_pass
            
        if cumulative_llr >= log_a:
            return {"status": "REJECT_FIX", "reason": "Test remains flaky", "runs": run_idx + 1}
        elif cumulative_llr <= log_b:
            return {"status": "ACCEPT_FIX", "reason": "Test confirmed clean", "runs": run_idx + 1}
    return {"status": "INCONCLUSIVE", "current_llr": cumulative_llr}
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
Design an autonomous agent verification contract that combines **Wald's SPRT**
with a **Hydraulic Shake Table (Synthetic Stress Injection)**:
- Under normal CI, the test has $\theta_1 = 0.020$.
- Under stress injection (CPU throttling + socket jitter), latent timing flaws
  are amplified $10\times$ to $\theta_{\text{stress}} = 0.200$.
- Calculate the new required clean runs and show how the agent saves $>80\%$
  of verification compute.

<details><summary><b>Click here to view the solution...</b></summary>

**Stress-Injected Agent Contract**:
```yaml
stress_testing_contract:
  target: "Accelerated Flaky Test Fix Qualification"
  harness_config:
    cpu_limit: "1 core"
    network_latency_ms: 150
    memory_pressure_mb: 512
  amplification_factor: 10.0
  parameters:
    theta_clean_stress: 0.005   # Clean code still rarely fails under stress
    theta_flaky_stress: 0.200   # Race condition fails 20% of the time under stress
    alpha: 0.05
    beta: 0.05
  sprt_calculations:
    delta_fail: "ln(0.200 / 0.005) = ln(40) = +3.689"  # > 2.94 (Aborts on first fail)
    delta_pass: "ln(0.800 / 0.995) = -0.218"
    runs_to_accept: "ceil(2.9444 / 0.218) = 14 runs"
  compute_savings:
    idle_runs_needed: 154
    stress_runs_needed: 14
    savings_percentage: "90.9% reduction in verification compute"
```
</details>



## Key Takeaways & Course Synthesis

1. **Fisher's Exact Test is Underpowered in CI**: Testing $2/100$ vs $0/100$
   yields
   $p = 0.497$. Do not let an AI assistant confuse lack of statistical power
   with evidence of failure.
2. **100 Passes is $BF = 4.0$**: 100 clean passes provides substantial evidence
   (updating a $60\%$ prior to $85.7\%$), but does not constitute proof beyond
   doubt.
3. **Wald's SPRT Eliminates Fixed Batches**: By evaluating evidence
   sequentially, SPRT halts on the very first failure ($\Delta \Lambda = +3.0$)
   and accepts clean fixes at $154$ runs.
4. **Stress Injection Slashes Compute**: Amplifying timing races via CPU
   throttling and network jitter reduces qualification runs from $154$ down to
   $14$, saving over $90\%$ of cloud compute.

---

## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 18: Flaky Tests & Pipeline Synthesis](18_case_studies_flaky_tests_and_pipelines.ipynb)** *(Part V: Production Reliability, CI/CD & Active Learning)*
* ➡️ **Next**: **[Ch 20: CI/CD Test Promotion Gatekeeping](20_test_promotion_heuristics_vs_bayes.ipynb)** *(Part V: Production Reliability, CI/CD & Active Learning)*
* 🔄 **Companion Tracks for Chapter 19**: [📖 Conceptual](../conceptual/19_verifying_flaky_test_fixes.ipynb) | [🤖 AI-Native Track](README.md) | [🐍 Python](../python/19_verifying_flaky_test_fixes.ipynb) | [🔵 R](../r/19_verifying_flaky_test_fixes.ipynb)

